In [1]:
!ls -al

total 1360
drwxrwxr-x. 3 junhyeong junhyeong   4096 Feb 10 12:21 .
drwxrwxr-x. 7 junhyeong junhyeong   4096 Dec 17 14:38 ..
-rw-rw-r--. 1 junhyeong junhyeong   3829 Dec 16 14:22 1-1.signalp.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  26272 Dec 16 16:07 1-2.pattern_match.ipynb
-rw-rw-r--. 1 junhyeong junhyeong   2609 Dec 16 16:40 1-3.drop_outlier.ipynb
-rw-rw-r--. 1 junhyeong junhyeong   9815 Dec 16 17:23 2-1.qc.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  94157 Dec 16 19:51 3-1.domain.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  10818 Dec 16 20:50 3-2.cluster.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  30124 Feb  5 13:57 3-3.Update_WY.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  22506 Dec 17 14:21 4-1.WY_select.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  11108 Dec 17 14:32 4-2.upgma.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  16667 Jan 28 15:49 4-3.cluster_by_branch_length.ipynb
-rw-rw-r--. 1 junhyeong junhyeong 129037 Jan 28 22:30 5-1.awr.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  15107 Jan 29 20:30 5-2.awr_se

In [2]:
%cd ../analysis/9.AWR-all-Phytophthora/

/var2/Works/junhyeong/RXLR_landscape/analysis/9.AWR-all-Phytophthora


/home/junhyeong/miniconda3/envs/biopython-env/lib/python3.13/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
import pandas as pd
from glob import glob
import os

In [4]:
!mkdir outputs

mkdir: cannot create directory ‘outputs’: File exists


In [5]:
data = glob("./proteome/*")

In [6]:
data[0]

'./proteome/Phybra.pep.faa'

In [7]:
for fasta in data:
    spp = os.path.basename(fasta).split(".")[0]
    cmd = f"conda run -n homology-env hmmsearch -o outputs/{spp}.final.results ../5.AWR/awr.final.hmm {fasta}"
    !{cmd}

# 데이터처리

In [8]:
import pandas as pd
from Bio import SearchIO, SeqIO

In [12]:
def parse_data(results, seq_db):
    wy1_query, wy2_query = SearchIO.parse(result, "hmmer3-text")
    
    wy1_hit = {hits.id: hits.hsps for hits in wy1_query.hits}
    wy2_hit = {hits.id: hits.hsps for hits in wy2_query.hits}

    awr_candidate = set(wy1_hit.keys()) | set(wy2_hit.keys())

    entry = []
    domain = []
    evalue = []
    start = []
    end = []
    seqs = []

    for hit in wy1_query.hits:
        for hsp in hit.hsps:
            if hsp.evalue < 1e-5:
                entry.append(hit.id)
                domain.append("wy1")
                evalue.append(hsp.evalue)
                start.append(hsp.hit_start)
                end.append(hsp.hit_end)
                seqs.append(seq_db[hit.id][hsp.hit_start:hsp.hit_end])

    for hit in wy2_query.hits:
        for hsp in hit.hsps:
            if hsp.evalue < 1e-5:
                entry.append(hit.id)
                domain.append("wy2")
                evalue.append(hsp.evalue)
                start.append(hsp.hit_start)
                end.append(hsp.hit_end)
                seqs.append(seq_db[hit.id][hsp.hit_start:hsp.hit_end])

    df = pd.DataFrame()
    df["entry"] = entry
    df["domain"] = domain
    df["evalue"] = evalue
    df["start"] = start
    df["end"] = end
    df["seq"] = seqs                

    return df

In [13]:
dfs = []

In [14]:
for fasta in data:
    cmd = f"echo {fasta}"
    !{cmd}
    
    spp = os.path.basename(fasta).split(".")[0]

    result = f"./outputs/{spp}.final.results"
    seq_db = {record.id: str(record.seq) for record in SeqIO.parse(f"./proteome/{spp}.pep.faa", "fasta")}

    df = parse_data(result, seq_db)
    df["spp"] = spp
    dfs.append(df)

./proteome/Phybra.pep.faa
./proteome/Phymul.pep.faa
./proteome/Phyram.pep.faa
./proteome/Plahal.pep.faa
./proteome/Phypav.pep.faa
./proteome/Phyinf.pep.faa
./proteome/Pytspi.pep.faa
./proteome/Hyabra.pep.faa
./proteome/Persor.pep.faa
./proteome/Phycap.pep.faa
./proteome/Phycit.pep.faa
./proteome/Phyuli.pep.faa
./proteome/Phyker.pep.faa
./proteome/Pytaph.pep.faa
./proteome/Phypin.pep.faa
./proteome/Pytult.pep.faa
./proteome/Phyplv.pep.faa
./proteome/Brelac.pep.faa
./proteome/Phyvig.pep.faa
./proteome/Phyboe.pep.faa
./proteome/Phyfol.pep.faa
./proteome/Phypar.pep.faa
./proteome/Phyhib.pep.faa
./proteome/Hyaara.pep.faa
./proteome/Phyvex.pep.faa
./proteome/Phycaj.pep.faa
./proteome/Phymel.pep.faa
./proteome/Phyale.pep.faa
./proteome/Phycry.pep.faa
./proteome/Pytoli.pep.faa
./proteome/Plavit.pep.faa
./proteome/Phypse.pep.faa
./proteome/Phyeur.pep.faa
./proteome/Phycin.pep.faa
./proteome/Pereff.pep.faa
./proteome/Phycac.pep.faa
./proteome/Phyaga.pep.faa
./proteome/Physyr.pep.faa
./proteome/P

# 데이터셋 합치기

In [16]:
awr_all = pd.concat(dfs)

In [44]:
awr_all = awr_all.sort_values(["entry", "end", "start"])

In [45]:
dom_all = awr_all[["entry", "domain"]].groupby("entry").aggregate(lambda x: '-'.join(x))

In [46]:
awr_all.to_csv("awr_all_search_results.parsed.tsv", sep = "\t", index = False)

In [47]:
awr_entries = set(dom_all.index)

In [48]:
count = awr_all[["entry", "domain"]].groupby("entry")["domain"].value_counts().unstack(fill_value = 0)

In [49]:
domain_count = pd.concat([dom_all, count], axis = 1)

In [50]:
dom_all.reset_index(inplace = True)

In [51]:
dom_all["spp"] = dom_all["entry"].map({entry: spp for entry, spp in zip(awr_all["entry"], awr_all["spp"])})

In [52]:
dom_all.sort_values(["spp", "entry"]).to_csv("dom_all.tsv", sep = "\t", index= False)

In [53]:
domain_count.to_csv("awr.tsv", sep = "\t")

In [ ]:
awr_entries = set(domain_count.index)

# 종별로 counting

In [59]:
dom_all.groupby(["spp", "domain"]).count().unstack().fillna(0).to_csv("temp.txt", sep = "\t")

In [60]:
dom_all

,entry,domain,spp
0,101354,wy1-wy2,Phycap
1,101579,wy1-wy2,Phycap
2,113929,wy1-wy2,Phycap
3,116585,wy1-wy1-wy1-wy1-wy1-wy2-wy1,Phycap
4,12657,wy1-wy2,Phycap
...,...,...,...
753,g657.t1,wy1-wy2,Phyplu
754,g8799.t1,wy1-wy2-wy1,Phyplu
755,g8805.t1,wy1-wy2-wy1,Phyplu
756,g9091.t1,wy1-wy2,Phyplu


# others

In [ ]:
with open("awr.fasta", "w") as f:
    for record in SeqIO.parse("../all_protein.faa", "fasta"):
        if record.id in awr_entries:
            f.write(f">{record.id}\n{record.seq}\n")